# Setting Up

In [1]:
import torch
import numpy as np
import pandas as pd
import random
from pathlib import Path
from torch.utils.data import DataLoader, Dataset, random_split
import sys
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
!nvidia-smi

Sat Jul 25 11:21:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.74                 KMD Version: 610.74        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P0             27W /  160W |       0MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Dataset

## Simple Approach

#### Most of the time in research we will have to create a custom Dataset class. The necessary functions and attributes are:
- __init__(self, ...): The Setup. This is where you pass in your raw data (or file paths). You store them as class attributes here.

- __len__(self): The Size. This function must return a single integer representing the total number of samples in your dataset. The DataLoader uses this to know when an epoch is over.

- __getitem__(self, idx): The Fetcher. This is the heart of the dataset. It takes an integer idx (index) and must return exactly one preprocessed sample (features and labels) converted to PyTorch Tensors.

In [7]:
dummy_data = {
    "sensor_1": np.random.rand(1000),
    "sensor_2": np.random.rand(1000),
    "sensor_3": np.random.rand(1000),
    "action_label": np.random.randint(0, 4, 1000) # 4 discrete RL actions
}
raw_df = pd.DataFrame(dummy_data)

In [13]:
class InMemoryDataset(Dataset):
    def __init__(self, data: pd.DataFrame, label_col: str):
        raw_features = data.drop(columns=[label_col]).values
        raw_labels = data[label_col].values

        self.features = torch.tensor(raw_features, dtype=torch.float32)
        self.labels = torch.tensor(raw_labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return {'features': self.features[idx], 'label': self.labels[idx]}
        

In [14]:
baseline_dataset = InMemoryDataset(raw_df, label_col="action_label")
sample = baseline_dataset[0]

print(f"\nSample keys: {sample.keys()}")
print(f"Features shape: {sample['features'].shape}, dtype: {sample['features'].dtype}")
print(f"Label value: {sample['label']}, dtype: {sample['label'].dtype}")


Sample keys: dict_keys(['features', 'label'])
Features shape: torch.Size([3]), dtype: torch.float32
Label value: 1, dtype: torch.int64


## The Scaling Bottleneck

### Proving the RAM Crash
Our Pandas-based `RealisticInMemoryDataset` works perfectly for a CSV with 10,000 rows. But what happens if we transition to Reinforcement Learning or Computer Vision? 

At a frontier lab, you might be training an agent on **1 Million Atari frames** or high-resolution MuJoCo camera feeds. Let's write a quick script to calculate exactly what happens to our system RAM if we try to load that dataset using our baseline approach.

In [16]:
def calculate_ram_usage(num_samples: int, shape: tuple, dtype_bytes: int = 4):
    """Calculates the theoretical RAM required to hold a dataset in memory."""
    elements_per_sample = 1
    for dim in shape:
        elements_per_sample *= dim
        
    total_bytes = num_samples * elements_per_sample * dtype_bytes
    return total_bytes / (1024 ** 3) # Convert to Gigabytes

# 1. Our previous CSV dataset (1,000 rows of 3 sensor features)
csv_gb = calculate_ram_usage(num_samples=1_000, shape=(3,))
print(f"RAM for 1k CSV rows: {csv_gb:.6f} GB (Safe!)")

# 2. A standard RL Vision Dataset (1 Million RGB Frames at 256x256)
# Shape: [Channels(3), Height(256), Width(256)]
vision_gb = calculate_ram_usage(num_samples=1_000_000, shape=(3, 256, 256))
print(f"RAM for 1M Atari frames: {vision_gb:.2f} GB (Unsafe!)")

RAM for 1k CSV rows: 0.000011 GB (Safe!)
RAM for 1M Atari frames: 732.42 GB (Unsafe!)


## The Scaling Fix (Lazy Loading)

**The Evidence:** Trying to load 1 million images into `__init__` requires **over 732 GB of RAM**. The Python process will instantly crash with an `Out Of Memory (OOM)` error.

The solution is Lazy Loading: 
1. **The Setup (`__init__`)**: We stop loading the actual data (payloads). Instead, we only load a list of string paths or memory indices (pointers).
2. **The Fetch (`__getitem__`)**: The heavy lifting is deferred here. We open the file from disk, convert it to a tensor, and return it *only when the DataLoader specifically asks for it*.

In [17]:
class LazyLoadDataset(Dataset):
    def __init__(self, paths:list[str], labels:list[int]):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # if each file is a single sample, this will work. If each file contains multiple samples, you will need to modify this logic accordingly.
        sample_path = self.paths[idx]
        sample_data = self.simulate_file_read(sample_path)  # Simulate reading the file
        return {'features': torch.tensor(sample_data, dtype=torch.float32),
                 'label': torch.tensor(self.labels[idx], dtype=torch.long)}
    def simulate_file_read(self, path):
        # Simulate reading a file and returning its content as a numpy array
        # In practice, you would read the actual file here.
        return np.random.rand(3, 256, 256)  # Simulating an RGB image of shape (3, 256, 256)
    

In [ ]:
dummy_paths = [f"data/frame_{i}.png" for i in range(1_000_000)]
dummy_labels = [np.random.randint(0, 4) for _ in range(1_000_000)]


In [23]:
lazy_dataset = LazyLoadDataset(dummy_paths, dummy_labels)
sample = lazy_dataset[0]
sample['features'].shape, sample['label']

(torch.Size([3, 256, 256]), tensor(3))

### Lets Download Dataset

In [3]:
TRAIN_DATA_DIR = Path('data/raw/train')
TRAIN_DATA_DIR.mkdir(parents=True, exist_ok=True)
TEST_DATA_DIR = Path('data/raw/test')
TEST_DATA_DIR.mkdir(parents=True, exist_ok=True)

###  Dataset

#### Best Practice 1

In [4]:
# Define a set seed function
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed) #note: this is for multi-GPU setups but for best practice

set_seed(SEED)
generator = torch.Generator().manual_seed(SEED)

In [5]:
train_subset_raw, val_subset_raw = random_split(
    raw_train_dataset, [45000, 5000], generator=generator
)

NameError: name 'raw_train_dataset' is not defined

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2  # Adjust based on CPU cores

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset_clean,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print(f"Train Batches : {len(train_loader)} (Batch size={BATCH_SIZE})")
print(f"Val Batches   : {len(val_loader)}")
print(f"Test Batches  : {len(test_loader)}")

## On The Fly Transformations (Augmentation)

### The Hard Drive Explosion

To prevent neural networks from memorizing training data, researchers use augmentation (e.g., adding Gaussian noise to RL state vectors, or randomly cropping images). 

**The Beginner Mistake (Static Augmentation):** A beginner might write a script that generates 5 augmented variations of every file and saves them all to the hard drive. 
* **The Math:** If your base dataset is 100 GB, it just became 600 GB. 
* **The Reality:** The neural network will eventually memorize those exact 5 static variations anyway.

**The Production Fix (On-The-Fly Transformations):**
Instead of saving variations to disk, we pass a callable function (a `transform`) into the Dataset's `__init__`. Inside `__getitem__`, *after* loading the raw data but *before* returning it to the DataLoader, we apply the function. This provides **infinite, non-repeating variations** while using **zero extra hard drive space**.

transform function can be used to transform data for any use. You can for instance use it for offline preprocessing (cleaning, feature engineering, etc.) or in our case right now we will be using it for online transformations (augmentation).

If your online transforms ever get too mathematically heavy and start slowing down the GPU, the advanced PyTorch move is to push the transforms out of the CPU Dataset entirely and execute them directly on the GPU batch using libraries like Kornia or torchvision.transforms.v2.

### Custom Transform

In [24]:
class AddGaussianNoise:
    def __init__(self, mean: float = 0.0, std: float = 0.1):
        self.mean = mean
        self.std = std

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        # torch.randn_like creates noise with the exact same shape and device as the input
        noise = torch.randn_like(tensor) * self.std + self.mean
        return tensor + noise

In [ ]:
class TransformedDataset(Dataset):
    def __init__(self, data_size: int, transform):
        self.data_size = data_size
        self.transform = transform  # Store the transform function

    def __len__(self) -> int:
        return self.data_size

    def __getitem__(self, idx: int):
        # 1. Simulate reading a raw state from disk
        # Let's pretend the true underlying state is a flat line of 1.0s
        raw_state = np.ones(5) 
        
        # 2. Strict Type Enforcement (From Step 2B)
        state_tensor = torch.tensor(raw_state, dtype=torch.float32)
        
        # 3. ⚡ ON-THE-FLY TRANSFORMATION ⚡
        if self.transform is not None:
            state_tensor = self.transform(state_tensor)
            
        return {
            "state": state_tensor,
            "label": torch.tensor(0, dtype=torch.int64) # Dummy label
        }